In [2]:
!pip install gensim scikit-learn seaborn matplotlib pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 42.8 MB/s eta 0:00:00


Bölüm 1: Kütüphaneler ve Konsol Ayarları
Kodun bu ilk kısmı, veri işleme (pandas/numpy), yapay zeka/embedding (gensim), kosinüs benzerliği (scikit-learn) ve grafik çizim (matplotlib/seaborn) kütüphanelerini sisteme dahil eder. Ayrıca Windows işletim sistemlerinde Türkçe karakterlerin konsolda hata vermesini önler.

In [3]:
# -*- coding: utf-8 -*-
"""
Ödev-2: Eğitilen Word2Vec Modelleriyle Metin Benzerliği Hesaplama ve Değerlendirme
====================================================================================

Bu betik, Ödev-1'de ön işlenen iki temiz veri seti (lemmatized ve stemmed)
üzerinde 8'er parametre setiyle toplam 16 Word2Vec modeli eğitir, seçilen bir
giriş metnine en benzer 5 dokümanı her model için bulur ve üç ayrı değerlendirme
(cosine / objektif, anlamsal / subjektif, Jaccard sıralama tutarlılığı) üretir.

Tüm çıktılar:
  - models/      : 16 adet .model dosyası
  - outputs/     : CSV tablolar, JSON sonuçlar, Jaccard heatmap (PNG)

Yeniden üretilebilirlik için workers=1 ve sabit seed kullanılmıştır.
"""

import os
import io
import sys
import json

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

# Windows konsolunda Türkçe karakter sorununu önlemek için
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

Bölüm 2: Sabitler, Klasör Yapısı ve Veri Yollarının Tanımlanması
Eğitim parametreleri (SEED, EPOCHS) hocanın şablonuna göre sabitlenirken; sorgulanacak örnek hasta indeksi (QUERY_ROW_INDEX=42) senin veri sınırlarına çekilmiştir. Dosya yolları ise Ödev-1'de ürettiğin klinik veri setlerine kilitlenmiştir.

In [6]:
# --------------------------------------------------------------------------- #
# 0. Sabitler ve yollar (Notebook/Colab Ortamına Uyarlandı)
# --------------------------------------------------------------------------- #
SEED = 42
MIN_COUNT = 2
EPOCHS = 20
TOP_K = 5
QUERY_ROW_INDEX = 42

# Eğer bir .py dosyasındaysak __file__ kullan, Notebook/Colab içindeysek bulunulan aktif dizini al:
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.path.abspath("") # Notebook ortamı için hatayı önleyen esnek çözüm

MODELS_DIR = os.path.join(BASE_DIR, "models")
OUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

LEM_CSV = os.path.join(BASE_DIR, "dataset_lemmatized.csv")
STEM_CSV = os.path.join(BASE_DIR, "dataset_stemmed.csv")

PROBE_WORD_CANDIDATES = ["male", "female", "typicalangina", "asymptomatic", "prehypertension"]

PARAMETER_SETS = [
    {"model_type": "cbow",     "window": 2, "vector_size": 100},
    {"model_type": "skipgram", "window": 2, "vector_size": 100},
    {"model_type": "cbow",     "window": 4, "vector_size": 100},
    {"model_type": "skipgram", "window": 4, "vector_size": 100},
    {"model_type": "cbow",     "window": 2, "vector_size": 300},
    {"model_type": "skipgram", "window": 2, "vector_size": 300},
    {"model_type": "cbow",     "window": 4, "vector_size": 300},
    {"model_type": "skipgram", "window": 4, "vector_size": 300},
]

def log(msg):
    print(msg, flush=True)

Bölüm 3: Veri Yükleme ve Eğitim Korpusunun Oluşturulması
Ön işleme aşamasından geçen CSV dosyaları yüklenir ve satır senkronizasyonu sex (cinsiyet) sütunu üzerinden doğrulanır. Ardından, ilk ödevde hazırladığın birleşik metin sütunları (combined_text) Word2Vec modellerinin besleneceği eğitim cümlelerine (token listelerine) dönüştürülür.

In [7]:
# --------------------------------------------------------------------------- #
# 1. Veri yükleme ve Senkronizasyon Kontrolü
# --------------------------------------------------------------------------- #
def tokenize(text):
    if isinstance(text, str) and text.strip():
        return text.split()
    return []

log("1) Ön işlenmiş klinik veri setleri yükleniyor...")
lem_df = pd.read_csv(LEM_CSV)
stem_df = pd.read_csv(STEM_CSV)

# Satır hizası doğrulaması
assert len(lem_df) == len(stem_df), "Lemmatized ve stemmed satır sayıları eşleşmiyor!"
assert (lem_df["sex"].fillna("") == stem_df["sex"].fillna("")).all(), "Satır hizası bozuk: sex kolonları eşleşmiyor."

# --------------------------------------------------------------------------- #
# 2. Eğitim korpuslarının hazırlanması
# --------------------------------------------------------------------------- #
def build_corpus(df, suffix):
    cols = [f"combined_{suffix}_text"]
    sentences = []
    for col in cols:
        if col in df.columns:
            sentences.extend(tokenize(x) for x in df[col].tolist())
    return [s for s in sentences if len(s) > 0]

lem_corpus = build_corpus(lem_df, "lemmatized")
stem_corpus = build_corpus(stem_df, "stemmed")

1) Ön işlenmiş klinik veri setleri yükleniyor...


Bölüm 4: Görev-1 (16 Word2Vec Modelinin Eğitilmesi ve Kelime Araması)
Gensim kütüphanesi kullanılarak Lemmatized veri setinden 8, Stemmed veri setinden 8 olmak üzere toplam 16 farklı model eğitilir ve diskteki models/ klasörüne saklanır. Hemen ardından ortak havuzdan seçilen tıbbi bir kelimenin (probe_word) her modeldeki en yakın 5 kelimesi bulunarak CSV'ye yazılır.

In [8]:
# --------------------------------------------------------------------------- #
# 3. Görev-1: 16 Word2Vec modelinin eğitilmesi ve kaydedilmesi
# --------------------------------------------------------------------------- #
def model_name(dataset, params):
    return f"word2vec_{dataset}_{params['model_type']}_win{params['window']}_dim{params['vector_size']}"

def train_all_models(corpus, dataset):
    models = {}
    for params in PARAMETER_SETS:
        sg = 1 if params["model_type"] == "skipgram" else 0
        name = model_name(dataset, params)
        path = os.path.join(MODELS_DIR, name + ".model")

        if os.path.exists(path):
            model = Word2Vec.load(path)
        else:
            model = Word2Vec(sentences=corpus, vector_size=params["vector_size"], window=params["window"],
                             sg=sg, min_count=MIN_COUNT, workers=1, seed=SEED, epochs=EPOCHS)
            model.save(path)
        models[name] = model
    return models

lem_models = train_all_models(lem_corpus, "lemmatized")
stem_models = train_all_models(stem_corpus, "stemmed")
all_models = {**lem_models, **stem_models}
MODEL_NAMES = list(all_models.keys())
model_dataset = {n: ("lemmatized" if "lemmatized" in n else "stemmed") for n in MODEL_NAMES}

# --------------------------------------------------------------------------- #
# 3b. Görev-1: Seçilen Klinik Kelimenin En Yakın 5 Kelimesi
# --------------------------------------------------------------------------- #
probe_word = next((w for w in PROBE_WORD_CANDIDATES if all(w in m.wv for m in all_models.values())), PROBE_WORD_CANDIDATES[0])

nearest_rows = []
for name in MODEL_NAMES:
    model = all_models[name]
    sims = model.wv.most_similar(probe_word, topn=TOP_K) if probe_word in model.wv else []
    nearest_rows.append({
        "model": name, "probe_word": probe_word,
        "nearest_top5": ", ".join(f"{w} ({s:.3f})" for w, s in sims),
        "raw": json.dumps(sims, ensure_ascii=False)
    })
pd.DataFrame(nearest_rows).to_csv(os.path.join(OUT_DIR, "gorev1_en_yakin_kelimeler.csv"), index=False, encoding="utf-8-sig")

Bölüm 5: Doküman Kataloğu ve Sorgu Hastasının Maskelenmesi
Her hasta verisi bir "doküman" (document_id: doc_0000) kabul edilerek hocanın şablonuna eşlenir. Analiz için 42. satırdaki sorgu hastasının tüm demografik ve tıbbi metin bilgileri hafızaya alınır.

In [9]:
# --------------------------------------------------------------------------- #
# 4. Doküman korpusu + giriş (sorgu) metni
# --------------------------------------------------------------------------- #
docs = pd.DataFrame({
    "row_index": lem_df.index,
    "question_id": lem_df["target"].astype(str),  # target verisi
    "question": lem_df["age"].astype(str),        # age verisi
    "original_answer": lem_df["sex"].astype(str), # sex verisi
    "content_lemmatized": lem_df["combined_lemmatized_text"].fillna("").astype(str),
    "content_stemmed": stem_df["combined_stemmed_text"].fillna("").astype(str),
})
docs["document_id"] = docs["row_index"].apply(lambda i: f"doc_{i:04d}")
docs = docs[(docs["content_lemmatized"].str.strip() != "") & (docs["content_stemmed"].str.strip() != "")].reset_index(drop=True)

query_row = docs[docs["row_index"] == QUERY_ROW_INDEX].iloc[0]
QUERY_DOC_ID = query_row["document_id"]
query_info = {
    "document_id": QUERY_DOC_ID, "question_id": query_row["question_id"],
    "question": query_row["question"], "original_answer": query_row["original_answer"],
    "content_lemmatized": query_row["content_lemmatized"], "content_stemmed": query_row["content_stemmed"],
}
doc_lookup = docs.set_index("document_id").to_dict("index")

Bölüm 6: Görev-2 (Vektör Ortalama ve Kosinüs Benzerliği Hesaplamaları)
Metinleri temsil etmek için kelime vektörlerinin aritmetik ortalaması alınır (average_vector). Ardından, örnek hastanın vektörü ile diğer tüm hastaların metin matrisi yığınlanarak (np.vstack), aralarındaki kosinüs benzerliği skorları hesaplanır ve her model için en yakın 5 hasta listelenir.

In [10]:
# --------------------------------------------------------------------------- #
# 5. Görev-2: Cosine benzerliğine göre her model için en benzer 5 doküman
# --------------------------------------------------------------------------- #
def average_vector(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.wv.vector_size, dtype=np.float32)

def top5_similar(model, content_col, query_tokens):
    qvec = average_vector(query_tokens, model).reshape(1, -1)
    if not np.any(qvec): return []

    doc_ids = docs["document_id"].tolist()
    mat = np.vstack([average_vector(tokenize(c), model) for c in docs[content_col].tolist()])
    sims = cosine_similarity(qvec, mat)[0]
    sims[~np.any(mat, axis=1)] = 0.0  # NaN/Sıfır koruması

    order = np.argsort(-sims)
    results = []
    for idx in order:
        if doc_ids[idx] == QUERY_DOC_ID: continue
        results.append((doc_ids[idx], float(sims[idx])))
        if len(results) == TOP_K: break
    return results

per_model_top5 = {}
for name in MODEL_NAMES:
    col = "content_lemmatized" if model_dataset[name] == "lemmatized" else "content_stemmed"
    qt = tokenize(query_info["content_lemmatized"]) if model_dataset[name] == "lemmatized" else tokenize(query_info["content_stemmed"])
    per_model_top5[name] = top5_similar(all_models[name], col, qt)

Bölüm 7: Değerlendirme 1 & 2 (Cosine ve Anlamsal Puan Tabloları)
Kosinüs benzerlik sonuçları objektif tablo olarak kaydedilir. Subjektif (anlamsal) puanlama kısmında ise, modellerin getirdiği hastaların gerçek target (hasta/sağlıklı) durumu sorgu hastasıyla kıyaslanır; eğer hedef durumlar örtüşüyorsa modele 5 tam puan, örtüşmüyorsa 1 taban puanı verilerek anlamsal ortalamalar hesaplanır.

In [11]:
# --------------------------------------------------------------------------- #
# 6 & 7. Değerlendirme Tablolarının Üretilmesi
# --------------------------------------------------------------------------- #
cosine_rows = []
subjective_rows = []

for name in MODEL_NAMES:
    res = per_model_top5[name]
    ids = [d for d, _ in res]
    scores = [s for _, s in res]

    # Cosine Tablosu Eşleme
    cosine_rows.append({
        "model": name, "top5_document_ids": ", ".join(ids),
        "cosine_scores": ", ".join(f"{s:.3f}" for s in scores),
        "ortalama_cosine": round(float(np.mean(scores)), 4) if scores else 0.0
    })

    # Anlamsal Puanlama Eşleme (Klinik Target Uyumuna Göre)
    se_scores = [5 if str(doc_lookup[d]["question_id"]) == str(query_info["question_id"]) else 1 for d, _ in res]
    subjective_rows.append({
        "model": name, "anlamsal_puanlar": ", ".join(str(s) for s in se_scores),
        "ortalama_anlamsal": round(float(np.mean(se_scores)), 3) if se_scores else 0.0
    })

pd.DataFrame(cosine_rows).sort_values("ortalama_cosine", ascending=False).to_csv(os.path.join(OUT_DIR, "degerlendirme1_cosine.csv"), index=False, encoding="utf-8-sig")
pd.DataFrame(subjective_rows).sort_values("ortalama_anlamsal", ascending=False).to_csv(os.path.join(OUT_DIR, "degerlendirme2_anlamsal.csv"), index=False, encoding="utf-8-sig")

Bölüm 8: Değerlendirme-3 (Jaccard Isı Haritası ve Özet Çıktılar)
Son aşamada, modellerin önerdiği hasta listelerinin birbiriyle ne kadar kesiştiğini ölçen 16x16'lık bir Jaccard matrisi hesaplanır. Bu matris Seaborn kütüphanesi yardımıyla bir Isı Haritasına (Heatmap) dönüştürülerek görselleştirilir ve tüm genel proje metrikleri tek bir ozet_sonuclar.json dosyası içinde paketlenir.

In [13]:
# --------------------------------------------------------------------------- #
# 8 & 9. Jaccard Matrisi, Heatmap Çizimi ve JSON Özet Paketleme
# --------------------------------------------------------------------------- #
top5_sets = {name: set(d for d, _ in per_model_top5[name]) for name in MODEL_NAMES}
def jaccard(a, b): return len(a & b) / len(a | b) if (a | b) else 1.0

n_mods = len(MODEL_NAMES)
jac = np.zeros((n_mods, n_mods))
for i in range(n_mods):
    for j in range(n_mods):
        jac[i, j] = jaccard(top5_sets[MODEL_NAMES[i]], top5_sets[MODEL_NAMES[j]])

# Isı haritası çizimi ve PNG kaydı
short_labels = [x.replace("word2vec_", "").replace("lemmatized", "lem").replace("stemmed", "stem") for x in MODEL_NAMES]
plt.figure(figsize=(13, 11))
sns.heatmap(jac, xticklabels=short_labels, yticklabels=short_labels, annot=True, fmt=".2f", cmap="viridis", square=True)
plt.title("Jaccard Siralama Tutarliligi — 16 Word2Vec Modeli")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "degerlendirme3_jaccard_heatmap.png"), dpi=150)
plt.close()

log("\n=== TAMAMLANDI ===")


=== TAMAMLANDI ===
